# T01 — Perbandingan Algoritma Machine Learning Klasik untuk Deteksi Pembukaan Lahan di Kawasan PSN Merauke

**Paket penelitian T01 · Blok A · Periode 2023 vs 2025**

Notebook ini menjalankan seluruh eksperimen T01 dari awal sampai akhir:

| Tahap | Isi | Keluaran |
|---|---|---|
| 1 | Menentukan Blok A secara otomatis (sel 20×20 km dengan kehilangan tutupan pohon 2024–2025 terbesar) | `07_dataset/T01_blokA.geojson` |
| 2 | Membangun label acuan konsensus dari Hansen GFC, JRC TMF dan RADD (**tanpa survei lapangan, tanpa data Sentinel-2**, sehingga tidak bocor ke fitur) | — |
| 3 | Ekstraksi 4 set fitur Sentinel-2 (spektral, indeks, multi-temporal, tekstur GLCM) | `07_dataset/T01_sampel.csv` |
| 4 | Pelatihan & tuning RF, SVM, GBT, KNN dengan validasi silang spasial (GroupKFold) | `03_tabel/*.csv` |
| 5 | Evaluasi: OA, Kappa, F1, waktu komputasi, uji McNemar | `03_tabel/*.csv`, `02_gambar/*.png` |
| 6 | Peta klasifikasi Blok A di Google Earth Engine + luas per kelas | `02_gambar/`, GeoTIFF di Google Drive |
| 7 | `T01_hasil_ringkas.json` → angka untuk mengganti placeholder **[HASIL]** di naskah | `03_tabel/` |

**Cara menjalankan (Google Colab):**
1. Buka https://colab.research.google.com → *File → Upload notebook* → pilih file ini.
2. Isi `GEE_PROJECT` di sel Konfigurasi dengan ID Cloud Project Earth Engine Anda (daftar gratis untuk riset: https://earthengine.google.com/noncommercial/).
3. *Runtime → Run all*. Saat diminta, login akun Google untuk Earth Engine.
4. Sel terakhir mengunduh `T01_hasil.zip` berisi dataset, tabel, dan gambar.

> **Integritas data:** semua angka di naskah wajib berasal dari keluaran notebook ini. Jangan mengubah atau mengarang angka.

In [ ]:
# 0. Instalasi pustaka (± 1 menit di Colab)
!pip install -q earthengine-api scikit-learn pandas matplotlib seaborn openpyxl requests

In [ ]:
# 1. Autentikasi Earth Engine
import ee
GEE_PROJECT = 'ISI-ID-PROJECT-ANDA'   # <-- GANTI, contoh: 'ee-namaanda'

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(force=True)
    ee.Initialize(project=GEE_PROJECT)
print('Earth Engine siap:', ee.String('OK').getInfo())

## Konfigurasi

In [ ]:
import os, json, time, math, random, io, zipfile
import numpy as np
import pandas as pd

TOPIC     = 'T01'
OUT       = f'/content/{TOPIC}_hasil' if os.path.exists('/content') else f'./{TOPIC}_hasil'
DIR_DATA  = os.path.join(OUT, '07_dataset')
DIR_TAB   = os.path.join(OUT, '03_tabel')
DIR_FIG   = os.path.join(OUT, '02_gambar')
for d in (DIR_DATA, DIR_TAB, DIR_FIG):
    os.makedirs(d, exist_ok=True)

SEED          = 42
PRE           = ('2023-01-01', '2023-12-31')   # sebelum pembukaan besar-besaran
POST          = ('2025-01-01', '2025-12-31')   # sesudah
BLOCK_KM      = 20                             # ukuran sel Blok
BLOCK_RANK    = 1                              # Blok A = peringkat 1 (B=2, C=3, D=4 dipakai topik lain)
N_PER_CLASS   = 700                            # jumlah sampel per kelas
CRS           = 'EPSG:32754'                   # UTM 54S
CLASS_NAMES   = {1: 'Lahan dibuka', 2: 'Hutan stabil', 3: 'Non-hutan stabil'}
DRIVE_FOLDER  = 'GEE_T01'

random.seed(SEED); np.random.seed(SEED)
print('Folder keluaran:', OUT)

## Tahap 1 — Wilayah studi: Kabupaten Merauke dan Blok A otomatis
Blok A dipilih secara objektif: grid 20×20 km di atas Kabupaten Merauke, lalu dipilih sel dengan luas kehilangan tutupan pohon Hansen 2024–2025 terbesar. Dengan cara ini tidak ada digitasi manual, dan topik lain dapat memakai Blok B/C/D (peringkat 2–4) sehingga wilayah sampel antar-paper tidak tumpang tindih.

In [ ]:
kab = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Merauke'))
kab_geom = kab.geometry()

gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
loss_recent = gfc.select('lossyear').gte(24).And(gfc.select('lossyear').lte(25))
loss_ha = ee.Image.pixelArea().divide(1e4).updateMask(loss_recent).rename('loss_ha')

grid = kab_geom.coveringGrid(ee.Projection(CRS).atScale(BLOCK_KM * 1000))
grid_stats = loss_ha.reduceRegions(collection=grid, reducer=ee.Reducer.sum(), scale=30, tileScale=8)
grid_sorted = grid_stats.filter(ee.Filter.gt('sum', 0)).sort('sum', False)

top = grid_sorted.limit(4).getInfo()['features']
blocks = pd.DataFrame([{'rank': i + 1, 'blok': 'ABCD'[i], 'loss_2024_2025_ha': round(f['properties']['sum'], 1)}
                       for i, f in enumerate(top)])
print(blocks.to_string(index=False))
blocks.to_csv(os.path.join(DIR_TAB, 'T01_tabel_kandidat_blok.csv'), index=False)

blok_feat = ee.Feature(ee.FeatureCollection(grid_sorted.toList(1, BLOCK_RANK - 1)).first())
blok = blok_feat.geometry()
with open(os.path.join(DIR_DATA, 'T01_blokA.geojson'), 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'blok': 'A'},
               'geometry': blok.transform('EPSG:4326', 1).getInfo()}]}, f)
c = blok.centroid(1).coordinates().getInfo()
print(f'Blok A — pusat: lon {c[0]:.4f}, lat {c[1]:.4f}; luas ± {BLOCK_KM*BLOCK_KM*100:,} ha')

## Tahap 2 — Fitur Sentinel-2 (pra & pasca)
Komposit median tahunan Sentinel-2 SR Harmonized dengan masking awan Cloud Score+. Empat set fitur dibandingkan:

| Set | Isi | Jumlah fitur |
|---|---|---|
| FS1 | 10 band spektral pasca (2025) | 10 |
| FS2 | FS1 + 6 indeks pasca (NDVI, EVI, NDWI, NDMI, NBR, BSI) | 16 |
| FS3 | FS2 + 6 indeks pra (2023) + 6 selisih indeks (pasca − pra) | 28 |
| FS4 | FS3 + 4 tekstur GLCM band NIR pasca | 32 |

In [ ]:
BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
IDX   = ['NDVI','EVI','NDWI','NDMI','NBR','BSI']
cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

def s2_composite(start, end, region):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(region).filterDate(start, end)
           .linkCollection(cs_plus, ['cs_cdf'])
           .map(lambda img: img.updateMask(img.select('cs_cdf').gte(0.6))
                                .select(BANDS).divide(10000)))
    return col.median().clip(region), col.size()

def add_indices(img):
    b = lambda n: img.select(n)
    ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
    evi  = img.expression('2.5*((N-R)/(N+6*R-7.5*B+1))', {'N': b('B8'), 'R': b('B4'), 'B': b('B2')}).rename('EVI')
    ndwi = img.normalizedDifference(['B3','B8']).rename('NDWI')
    ndmi = img.normalizedDifference(['B8','B11']).rename('NDMI')
    nbr  = img.normalizedDifference(['B8','B12']).rename('NBR')
    bsi  = img.expression('((S+R)-(N+B))/((S+R)+(N+B))', {'S': b('B11'), 'R': b('B4'), 'N': b('B8'), 'B': b('B2')}).rename('BSI')
    return ee.Image.cat([ndvi, evi, ndwi, ndmi, nbr, bsi])

region = blok.buffer(500)
post, n_post = s2_composite(*POST, region)
pre,  n_pre  = s2_composite(*PRE,  region)
idx_post = add_indices(post)
idx_pre  = add_indices(pre)

post_b   = post.rename([f'{x}_post' for x in BANDS])
idxp     = idx_post.rename([f'{x}_post' for x in IDX])
idxq     = idx_pre.rename([f'{x}_pre' for x in IDX])
idxd     = idx_post.subtract(idx_pre).rename([f'd{x}' for x in IDX])
glcm     = (post.select('B8').multiply(10000).toInt32().glcmTexture(size=3)
            .select(['B8_contrast','B8_ent','B8_savg','B8_var'])
            .rename(['GLCM_contrast','GLCM_entropy','GLCM_savg','GLCM_var']))
features = ee.Image.cat([post_b, idxp, idxq, idxd, glcm]).toFloat()

FS = {
    'FS1': [f'{x}_post' for x in BANDS],
    'FS2': [f'{x}_post' for x in BANDS] + [f'{x}_post' for x in IDX],
    'FS3': [f'{x}_post' for x in BANDS] + [f'{x}_post' for x in IDX] + [f'{x}_pre' for x in IDX] + [f'd{x}' for x in IDX],
}
FS['FS4'] = FS['FS3'] + ['GLCM_contrast','GLCM_entropy','GLCM_savg','GLCM_var']
print('Jumlah citra S2 — pra 2023:', n_pre.getInfo(), '| pasca 2025:', n_post.getInfo())

## Tahap 3 — Label acuan konsensus (tanpa survei lapangan)
Label dibangun **hanya dari produk independen berbasis Landsat dan Sentinel-1**, bukan dari Sentinel-2, agar tidak terjadi kebocoran informasi (*label leakage*) ke fitur.

| Kelas | Aturan |
|---|---|
| 1 Lahan dibuka | Hansen `lossyear` ∈ {2024, 2025} **dan** (RADD alert 2024–2025 **atau** JRC TMF Des-2025 = deforestasi/degradasi) |
| 2 Hutan stabil | Hansen tutupan pohon 2000 ≥ 50% **dan** tanpa kehilangan 2001–2025 **dan** tanpa alert RADD |
| 3 Non-hutan stabil | Hansen tutupan pohon 2000 < 10% **dan** tanpa kehilangan **dan** tanpa alert RADD **dan** bukan air (JRC GSW occurrence < 50%) |

Catatan: aturan awal berbasis kelas TMF untuk kelas stabil menghasilkan 0 piksel di Blok A, karena hutan *Melaleuca*/musim dan savana Merauke tidak terwakili sebagai hutan tropis lembap. Piksel air dan piksel yang tidak memenuhi aturan mana pun tidak diberi label.

In [ ]:
tmf = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
tmf23, tmf25 = tmf.select('Dec2023'), tmf.select('Dec2025')

radd = (ee.ImageCollection('projects/radar-wur/raddalert/v1')
        .filterMetadata('layer', 'contains', 'alert')
        .filterMetadata('geography', 'equals', 'asia')
        .sort('system:time_end', False).first())
radd_date = radd.select('Date')                       # format YYDOY
radd_hit  = radd_date.gte(24001).And(radd_date.lte(25366)).unmask(0)

lossyear  = gfc.select('lossyear').unmask(0)   # v1.13: piksel tanpa loss bernilai masked
tc2000    = gfc.select('treecover2000')
no_loss   = lossyear.eq(0)

c1 = lossyear.gte(24).And(lossyear.lte(25)).And(radd_hit.Or(tmf25.eq(3)).Or(tmf25.eq(2)))
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0)
c2 = tc2000.gte(50).And(no_loss).And(radd_hit.eq(0))                  # hutan stabil
c3 = tc2000.lt(10).And(no_loss).And(radd_hit.eq(0)).And(gsw.lt(50))   # non-hutan stabil (bukan air)

label = (ee.Image(0).where(c3, 3).where(c2, 2).where(c1, 1)
         .updateMask(gsw.lt(50)).selfMask().rename('label').toInt())

# Jumlah piksel kandidat per kelas (30 m) — untuk melaporkan ketersediaan label
hist = label.reduceRegion(ee.Reducer.frequencyHistogram(), blok, 30, maxPixels=1e10, tileScale=8).get('label').getInfo()
avail = pd.DataFrame([{'kelas': CLASS_NAMES[int(k)], 'piksel_30m': v, 'luas_ha': round(v * 0.09, 1)}
                      for k, v in sorted(hist.items(), key=lambda kv: int(kv[0]))])
print(avail.to_string(index=False))
avail.to_csv(os.path.join(DIR_TAB, 'T01_tabel_ketersediaan_label.csv'), index=False)

## Tahap 4 — Pengambilan sampel acak berstrata & dataset

In [ ]:
sample_img = label.addBands(features)
fc = sample_img.stratifiedSample(numPoints=N_PER_CLASS, classBand='label', region=blok, scale=10,
                                 seed=SEED, geometries=True, tileScale=8, dropNulls=True)

def fc_to_df(fc, page=2000):
    n = fc.size().getInfo()
    lst = fc.toList(n)
    rows = []
    for start in range(0, n, page):
        feats = ee.FeatureCollection(lst.slice(start, start + page)).getInfo()['features']
        for f in feats:
            p = dict(f['properties'])
            p['lon'], p['lat'] = f['geometry']['coordinates']
            rows.append(p)
    return pd.DataFrame(rows)

t0 = time.perf_counter()
df = fc_to_df(fc).dropna()
df['label'] = df['label'].astype(int)
df['kelas'] = df['label'].map(CLASS_NAMES)
# Kelompok spasial 2 km untuk validasi silang spasial (mencegah titik bertetangga masuk train & test)
df['grup_spasial'] = (np.floor(df['lon'] / 0.018).astype(int).astype(str) + '_' +
                      np.floor(df['lat'] / 0.018).astype(int).astype(str))
print(f'{len(df)} sampel dalam {time.perf_counter()-t0:.0f} dtk')
print(df['kelas'].value_counts())

df.to_csv(os.path.join(DIR_DATA, 'T01_sampel.csv'), index=False)
gj = {'type': 'FeatureCollection', 'features': [
      {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [r.lon, r.lat]},
       'properties': {'id': int(i), 'label': int(r.label), 'kelas': r.kelas}} for i, r in df.iterrows()]}
with open(os.path.join(DIR_DATA, 'T01_sampel_titik.geojson'), 'w') as f:
    json.dump(gj, f)

## Tahap 5 — Eksperimen: 4 algoritma × 4 set fitur
* Pembagian data: 70% latih / 30% uji dengan **GroupShuffleSplit** berdasarkan sel spasial 2 km.
* Tuning hiperparameter: **GridSearchCV** dengan **GroupKFold (k = 5)** pada data latih, metrik F1-makro.
* Model pembanding juga dijalankan dengan parameter bawaan (*default*) untuk mengukur dampak tuning.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, cohen_kappa_score, f1_score, confusion_matrix,
                             precision_recall_fscore_support)

def make_models():
    return {
        'RF':  (Pipeline([('clf', RandomForestClassifier(random_state=SEED, n_jobs=-1))]),
                {'clf__n_estimators': [100, 300, 500], 'clf__max_features': ['sqrt', 0.3],
                 'clf__min_samples_leaf': [1, 3]}),
        'SVM': (Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='rbf', random_state=SEED))]),
                {'clf__C': [1, 10, 100], 'clf__gamma': ['scale', 0.01, 0.1]}),
        'GBT': (Pipeline([('clf', GradientBoostingClassifier(random_state=SEED))]),
                {'clf__n_estimators': [100, 300], 'clf__learning_rate': [0.05, 0.1], 'clf__max_depth': [3, 5]}),
        'KNN': (Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier())]),
                {'clf__n_neighbors': [3, 5, 7, 11, 15], 'clf__weights': ['uniform', 'distance']}),
    }

def split_data(df, seed=SEED):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr, te = next(gss.split(df, df['label'], df['grup_spasial']))
    return df.iloc[tr].reset_index(drop=True), df.iloc[te].reset_index(drop=True)

def evaluate(y_true, y_pred):
    return {'OA': accuracy_score(y_true, y_pred),
            'Kappa': cohen_kappa_score(y_true, y_pred),
            'F1_makro': f1_score(y_true, y_pred, average='macro')}

def run_experiment(df, feature_sets, models_fn=make_models, cv_k=5):
    train, test = split_data(df)
    rows, preds, best_params, fitted = [], {}, {}, {}
    for fs_name, cols in feature_sets.items():
        Xtr, ytr, gtr = train[cols].values, train['label'].values, train['grup_spasial'].values
        Xte, yte = test[cols].values, test['label'].values
        for m_name, (pipe, grid) in models_fn().items():
            # (a) parameter bawaan
            t0 = time.perf_counter(); pipe.fit(Xtr, ytr); t_fit_def = time.perf_counter() - t0
            yp_def = pipe.predict(Xte)
            rows.append({'set_fitur': fs_name, 'n_fitur': len(cols), 'model': m_name, 'konfigurasi': 'default',
                         **evaluate(yte, yp_def), 'waktu_latih_dtk': t_fit_def, 'param_terbaik': '-'})
            # (b) hasil tuning
            gs = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=cv_k), scoring='f1_macro', n_jobs=-1)
            t0 = time.perf_counter(); gs.fit(Xtr, ytr, groups=gtr); t_tune = time.perf_counter() - t0
            t0 = time.perf_counter(); yp = gs.predict(Xte); t_pred = time.perf_counter() - t0
            rows.append({'set_fitur': fs_name, 'n_fitur': len(cols), 'model': m_name, 'konfigurasi': 'tuned',
                         **evaluate(yte, yp), 'waktu_latih_dtk': t_tune, 'waktu_prediksi_dtk': t_pred,
                         'F1_cv': gs.best_score_,
                         'param_terbaik': json.dumps({k.replace('clf__', ''): v for k, v in gs.best_params_.items()})})
            preds[(fs_name, m_name)] = yp
            best_params[(fs_name, m_name)] = gs.best_params_
            fitted[(fs_name, m_name)] = gs.best_estimator_
            print(f'{fs_name} {m_name:3s}  OA={rows[-1]["OA"]:.3f}  Kappa={rows[-1]["Kappa"]:.3f}  F1={rows[-1]["F1_makro"]:.3f}')
    return pd.DataFrame(rows), train, test, preds, best_params, fitted

results, train, test, preds, best_params, fitted = run_experiment(df, FS)
results.to_csv(os.path.join(DIR_TAB, 'T01_tabel_hasil_lengkap.csv'), index=False)
print(f'\nLatih: {len(train)} sampel | Uji: {len(test)} sampel')

## Tahap 6 — Seleksi fitur (permutation importance) dan uji signifikansi McNemar

In [ ]:
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

# Permutation importance dihitung HANYA dari data latih (split validasi internal berbasis grup spasial),
# agar seleksi fitur FS5 tidak melihat data uji (mencegah kebocoran/optimistic bias)
from sklearn.model_selection import GroupShuffleSplit as _GSS
itr, iva = next(_GSS(n_splits=1, test_size=0.25, random_state=SEED).split(train, train['label'], train['grup_spasial']))
tr_in, va_in = train.iloc[itr], train.iloc[iva]
rf_bp = {k.replace('clf__', ''): v for k, v in best_params[('FS4', 'RF')].items()}
rf_in = RandomForestClassifier(random_state=SEED, n_jobs=-1, **rf_bp).fit(tr_in[FS['FS4']].values, tr_in['label'].values)
pi = permutation_importance(rf_in, va_in[FS['FS4']].values, va_in['label'].values, n_repeats=20,
                            random_state=SEED, scoring='f1_macro', n_jobs=-1)
imp = (pd.DataFrame({'fitur': FS['FS4'], 'importance_mean': pi.importances_mean, 'importance_std': pi.importances_std})
       .sort_values('importance_mean', ascending=False).reset_index(drop=True))
imp.to_csv(os.path.join(DIR_TAB, 'T01_tabel_permutation_importance.csv'), index=False)

# FS5 = 10 fitur teratas, lalu dievaluasi ulang dengan 4 algoritma
FS5 = {'FS5': imp['fitur'].head(10).tolist()}
res5, _, _, preds5, bp5, fit5 = run_experiment(df, FS5)
results_all = pd.concat([results, res5], ignore_index=True)
preds.update(preds5); best_params.update(bp5); fitted.update(fit5)
results_all.to_csv(os.path.join(DIR_TAB, 'T01_tabel_hasil_lengkap.csv'), index=False)
print('FS5 =', FS5['FS5'])

tuned = results_all[results_all['konfigurasi'] == 'tuned'].copy()
best_row = tuned.sort_values(['F1_makro', 'OA'], ascending=False).iloc[0]
BEST_FS, BEST_MODEL = best_row['set_fitur'], best_row['model']
print(f'Kombinasi terbaik: {BEST_MODEL} + {BEST_FS}  (F1-makro {best_row["F1_makro"]:.3f})')

def mcnemar_exact(y, a, b):
    # Uji McNemar eksak (binomial) antara prediksi model a dan b
    a_ok, b_ok = (a == y), (b == y)
    n01 = int(np.sum(a_ok & ~b_ok)); n10 = int(np.sum(~a_ok & b_ok))
    p = binomtest(n01, n01 + n10, 0.5).pvalue if (n01 + n10) > 0 else 1.0
    return n01, n10, p

yte = test['label'].values
mods = ['RF', 'SVM', 'GBT', 'KNN']
mc = []
for i in range(len(mods)):
    for j in range(i + 1, len(mods)):
        n01, n10, p = mcnemar_exact(yte, preds[(BEST_FS, mods[i])], preds[(BEST_FS, mods[j])])
        mc.append({'set_fitur': BEST_FS, 'model_A': mods[i], 'model_B': mods[j],
                   'A_benar_B_salah': n01, 'A_salah_B_benar': n10, 'p_value': p, 'signifikan_0.05': p < 0.05})
mcn = pd.DataFrame(mc)
mcn.to_csv(os.path.join(DIR_TAB, 'T01_tabel_mcnemar.csv'), index=False)
print(mcn.to_string(index=False))

## Tahap 7 — Tabel ringkas dan gambar untuk naskah

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'figure.dpi': 110})

tuned_all = results_all[results_all['konfigurasi'] == 'tuned']

# Tabel utama: F1-makro per model × set fitur
pivot_f1 = tuned_all.pivot(index='model', columns='set_fitur', values='F1_makro').loc[mods]
pivot_oa = tuned_all.pivot(index='model', columns='set_fitur', values='OA').loc[mods]
pivot_f1.round(4).to_csv(os.path.join(DIR_TAB, 'T01_tabel_F1_model_x_fitur.csv'))
pivot_oa.round(4).to_csv(os.path.join(DIR_TAB, 'T01_tabel_OA_model_x_fitur.csv'))

# Dampak tuning (set fitur terbaik)
cmp = (results_all[results_all['set_fitur'] == BEST_FS]
       .pivot(index='model', columns='konfigurasi', values='F1_makro').loc[mods])
cmp['selisih'] = cmp['tuned'] - cmp['default']
cmp.round(4).to_csv(os.path.join(DIR_TAB, 'T01_tabel_dampak_tuning.csv'))

# Metrik per kelas untuk model terbaik
yp_best = preds[(BEST_FS, BEST_MODEL)]
p, r, f, s = precision_recall_fscore_support(yte, yp_best, labels=[1, 2, 3])
per_class = pd.DataFrame({'kelas': [CLASS_NAMES[k] for k in [1, 2, 3]], 'presisi (UA)': p,
                          'recall (PA)': r, 'F1': f, 'n_uji': s})
per_class.round(4).to_csv(os.path.join(DIR_TAB, 'T01_tabel_metrik_per_kelas.csv'), index=False)

# Gambar 1: heatmap F1
fig, ax = plt.subplots(figsize=(6.5, 3.2))
sns.heatmap(pivot_f1, annot=True, fmt='.3f', cmap='Greens', cbar_kws={'label': 'F1-makro'}, ax=ax)
ax.set_xlabel('Set fitur'); ax.set_ylabel('Algoritma')
fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, 'T01_gambar_heatmap_F1.png'), dpi=300); plt.show()

# Gambar 2: matriks konfusi 4 algoritma (set fitur terbaik)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
short = ['Dibuka', 'Hutan', 'Non-hutan']
for ax, m in zip(axes, mods):
    cm = confusion_matrix(yte, preds[(BEST_FS, m)], labels=[1, 2, 3])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, xticklabels=short, yticklabels=short)
    ax.set_title(f'{m} ({BEST_FS})'); ax.set_xlabel('Prediksi'); ax.set_ylabel('Acuan')
fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, 'T01_gambar_matriks_konfusi.png'), dpi=300); plt.show()

# Gambar 3: permutation importance 15 teratas
top15 = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.barh(top15['fitur'], top15['importance_mean'], xerr=top15['importance_std'], color='#2e7d32')
ax.set_xlabel('Penurunan F1-makro (permutation importance)')
fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, 'T01_gambar_importance.png'), dpi=300); plt.show()

# Gambar 4: waktu latih vs F1
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for m, mk in zip(mods, ['o', 's', '^', 'D']):
    d = tuned_all[tuned_all['model'] == m]
    ax.scatter(d['waktu_latih_dtk'], d['F1_makro'], marker=mk, s=45, label=m)
ax.set_xscale('log'); ax.set_xlabel('Waktu tuning + latih (detik, skala log)'); ax.set_ylabel('F1-makro')
ax.legend(frameon=False); fig.tight_layout()
fig.savefig(os.path.join(DIR_FIG, 'T01_gambar_waktu_vs_F1.png'), dpi=300); plt.show()
print(pivot_f1.round(3))

## Tahap 8 — Peta klasifikasi Blok A di Google Earth Engine
Peta dibuat dengan padanan Earth Engine dari model berbasis pohon (RF atau GBT) yang terbaik, memakai hiperparameter hasil tuning dan set fitur terbaik. SVM dan KNN tidak dipetakan karena memerlukan standardisasi yang tidak identik di Earth Engine; hal ini dicatat sebagai batasan.

In [ ]:
import requests

tree_rows = tuned_all[(tuned_all['set_fitur'] == BEST_FS) & (tuned_all['model'].isin(['RF', 'GBT']))]
MAP_MODEL = tree_rows.sort_values('F1_makro', ascending=False).iloc[0]['model']
bp = {k.replace('clf__', ''): v for k, v in best_params[(BEST_FS, MAP_MODEL)].items()}
cols = (FS5['FS5'] if BEST_FS == 'FS5' else FS[BEST_FS])

if MAP_MODEL == 'RF':
    mf = bp['max_features']
    vps = int(round(math.sqrt(len(cols)))) if mf == 'sqrt' else max(1, int(round(mf * len(cols))))
    clf = ee.Classifier.smileRandomForest(numberOfTrees=int(bp['n_estimators']), variablesPerSplit=vps,
                                          minLeafPopulation=int(bp['min_samples_leaf']), seed=SEED)
else:
    clf = ee.Classifier.smileGradientTreeBoost(numberOfTrees=int(bp['n_estimators']),
                                               shrinkage=float(bp['learning_rate']),
                                               maxNodes=int(2 ** int(bp['max_depth'])), seed=SEED)

train_fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(r['lon']), float(r['lat'])]),
                                            {'label': int(r['label']), **{c: float(r[c]) for c in cols}})
                                 for _, r in train.iterrows()])
clf = clf.train(train_fc, 'label', cols)
classified = features.select(cols).classify(clf).clip(blok).rename('kelas')

area = (ee.Image.pixelArea().divide(1e4).addBands(classified)
        .reduceRegion(ee.Reducer.sum().group(1, 'kelas'), blok, 10, maxPixels=1e11, tileScale=8).get('groups').getInfo())
luas = pd.DataFrame([{'kelas': CLASS_NAMES.get(int(g['kelas']), str(g['kelas'])), 'luas_ha': round(g['sum'], 1)} for g in area])
luas['persen'] = (100 * luas['luas_ha'] / luas['luas_ha'].sum()).round(2)
luas.to_csv(os.path.join(DIR_TAB, 'T01_tabel_luas_per_kelas.csv'), index=False)
print('Model peta:', MAP_MODEL, '| set fitur:', BEST_FS); print(luas.to_string(index=False))

def save_thumb(img, vis, path, dim=800):
    # reproject 30 m khusus untuk tampilan, agar tidak 'User memory limit exceeded'
    url = img.reproject(crs=CRS, scale=30).getThumbURL({**vis, 'region': blok, 'dimensions': dim, 'format': 'png'})
    r = requests.get(url, timeout=300)
    with open(path, 'wb') as f:
        f.write(r.content)
    print(os.path.basename(path), len(r.content), 'byte', '(GAGAL)' if len(r.content) < 1000 else 'OK')

save_thumb(classified, {'min': 1, 'max': 3, 'palette': ['#d73027', '#1a9850', '#fee08b']},
           os.path.join(DIR_FIG, 'T01_gambar_peta_klasifikasi.png'))
save_thumb(pre,  {'bands': ['B4','B3','B2'], 'min': 0, 'max': 0.15}, os.path.join(DIR_FIG, 'T01_gambar_S2_2023.png'))
save_thumb(post, {'bands': ['B4','B3','B2'], 'min': 0, 'max': 0.15}, os.path.join(DIR_FIG, 'T01_gambar_S2_2025.png'))

# Peta lokasi: Kabupaten Merauke + Blok A
loc = ee.Image().paint(kab, 0, 2).visualize(palette=['333333']).blend(
      ee.Image().paint(ee.FeatureCollection([blok_feat]), 0, 4).visualize(palette=['d73027']))
url = loc.getThumbURL({'region': kab_geom.bounds(), 'dimensions': 1200, 'format': 'png'})
with open(os.path.join(DIR_FIG, 'T01_gambar_lokasi.png'), 'wb') as f:
    f.write(requests.get(url, timeout=300).content)

# GeoTIFF resolusi penuh ke Google Drive (opsional, lihat tab Tasks di code.earthengine.google.com)
task = ee.batch.Export.image.toDrive(image=classified.toByte(), description='T01_peta_klasifikasi_blokA',
                                     folder=DRIVE_FOLDER, region=blok, scale=10, crs=CRS, maxPixels=1e11)
task.start(); print('Ekspor GeoTIFF dimulai:', task.status()['state'])

## Tahap 9 — Ringkasan angka untuk naskah dan unduh semua keluaran

In [ ]:
summary = {
    'blok_A_loss_2024_2025_ha': float(blocks.iloc[BLOCK_RANK - 1]['loss_2024_2025_ha']),
    'jumlah_citra_S2_pra': n_pre.getInfo(), 'jumlah_citra_S2_pasca': n_post.getInfo(),
    'jumlah_sampel_total': int(len(df)), 'jumlah_sampel_latih': int(len(train)), 'jumlah_sampel_uji': int(len(test)),
    'sampel_per_kelas': df['kelas'].value_counts().to_dict(),
    'kombinasi_terbaik': {'model': BEST_MODEL, 'set_fitur': BEST_FS, **{k: round(float(best_row[k]), 4) for k in ['OA', 'Kappa', 'F1_makro']}},
    'F1_makro_model_x_fitur': pivot_f1.round(4).to_dict(),
    'OA_model_x_fitur': pivot_oa.round(4).to_dict(),
    'dampak_tuning_F1': cmp['selisih'].round(4).to_dict(),
    'FS5_10_fitur_teratas': FS5['FS5'],
    'mcnemar': mcn[['model_A', 'model_B', 'p_value']].round(4).to_dict('records'),
    'model_peta': MAP_MODEL, 'luas_per_kelas_ha': luas.set_index('kelas')['luas_ha'].to_dict(),
}
with open(os.path.join(DIR_TAB, f'{TOPIC}_hasil_ringkas.json'), 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

# Excel berisi semua tabel
with pd.ExcelWriter(os.path.join(DIR_TAB, f'{TOPIC}_semua_tabel.xlsx')) as xw:
    for fn in sorted(os.listdir(DIR_TAB)):
        if fn.endswith('.csv'):
            pd.read_csv(os.path.join(DIR_TAB, fn)).to_excel(xw, sheet_name=fn.replace(f'{TOPIC}_tabel_', '')[:31].replace('.csv', ''), index=False)

zpath = f'{OUT}.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), os.path.dirname(OUT)))
print(json.dumps(summary, indent=2, ensure_ascii=False, default=str))
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Arsip tersimpan di', zpath)

## Langkah berikutnya
1. Ekstrak `T01_hasil.zip` ke folder paket T01: `02_gambar/`, `03_tabel/`, `07_dataset/`.
2. Buka naskah `01_naskah/T01_naskah_ID.docx` dan `T01_manuscript_EN.docx`, lalu ganti setiap **[HASIL: …]** dengan angka dari `T01_hasil_ringkas.json` dan tabel CSV.
3. Sisipkan gambar dari `02_gambar/` pada posisi yang ditandai **[GAMBAR: …]**.
4. Tulis ulang paragraf Pembahasan berdasarkan angka yang benar-benar diperoleh.